# AWREAS — Google Colab Backend Launcher (T4 GPU Edition)

> **Instructions:**
> 1. Go to `Runtime` → `Change runtime type` → select **T4 GPU**
> 2. Add your API keys as Colab Secrets (🔑 icon on the left sidebar)
> 3. Run cells top-to-bottom
> 4. Copy the 🔥 PUBLIC API URL and use it in your local benchmark scripts

In [2]:
# ── Cell 1: Verify GPU is available ──────────────────────────────────────────
import torch
print(f'CUDA Available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

CUDA Available : False
⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU


In [3]:
# ── Cell 2: Install all required dependencies ─────────────────────────────────
!pip install -q \
    fastapi uvicorn[standard] python-multipart \
    pyngrok nest_asyncio \
    google-genai \
    pydantic pydantic-settings \
    sqlalchemy[asyncio] aiosqlite asyncpg alembic \
    httpx \
    pypdf pymupdf python-docx \
    torch transformers sentence-transformers \
    spacy

# Download the spaCy English model used by CitationWorker
!python -m spacy download en_core_web_sm -q

print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 2.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 4.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 19.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.3 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Dependencies installed


## Step 2 — Upload Project Code

Zip the `academic-review-system` folder on your PC:
```powershell
# Run this in PowerShell from your Desktop
Compress-Archive -Path .\academic-review-system\* -DestinationPath academic-review-system.zip
```
Then upload the zip here using the file panel (📁) on the left.

In [ ]:
# ── Cell 3: Unzip project & confirm structure ─────────────────────────────────
!unzip -q -o academic-review-system.zip -d /content/awreas
import os
os.chdir('/content/awreas')
!ls -la   # should show main.py, app/, scripts/, data/, etc.

In [ ]:
# ── Cell 4: Secrets / API Keys ────────────────────────────────────────────────
# Best practice: add GEMINI_API_KEY and NGROK_TOKEN as Colab Secrets
# (click the 🔑 key icon in the left sidebar → New Secret)
# Then retrieve them safely:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')   # add this in Colab Secrets
NGROK_TOKEN    = userdata.get('NGROK_TOKEN')       # add this in Colab Secrets

# Write .env so the app settings loader can pick it up
env_content = f"""
GEMINI_API_KEY={GEMINI_API_KEY}
LLM_PROVIDER=gemini
USE_LLM_SYNTHESIS=true
ENABLE_WEB_RESEARCH=false
DATABASE_URL=sqlite+aiosqlite:///./awreas_colab.db
"""
with open('.env', 'w') as f:
    f.write(env_content.strip())

print('✅ Environment configured')

In [ ]:
# ── Cell 5: Setup Ngrok tunnel ────────────────────────────────────────────────
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url

print(f'\n🔥 PUBLIC API URL: {public_url} 🔥\n')
print('── Copy the command below and run it on your LOCAL PC ─────────────────')
print(f'python scripts/benchmark_comparers.py \\')
print(f'    --dataset data/asap/subset_test.jsonl \\')
print(f'    --system "AWREAS_Full={public_url}/api/v1/evaluate" \\')
print(f'    --runs 1 \\')
print(f'    --timeout 120 \\')
print(f'    --out-json data/asap/benchmark_results_prompt1.json \\')
print(f'    --out-md   data/asap/benchmark_report_prompt1.md')

In [ ]:
# ── Cell 6: Run database migrations ──────────────────────────────────────────
# Only needed once per Colab session. SQLite is used in Colab for simplicity.
!python -m alembic upgrade head 2>/dev/null || echo 'Alembic skipped (no migrations needed or DB already up to date)'

In [ ]:
# ── Cell 7: Patch lifespan & launch server ────────────────────────────────────
import os
import nest_asyncio
import uvicorn
from contextlib import asynccontextmanager
from fastapi import FastAPI

from main import app
from app.llm import LLMClient
from app.supervisor.agent import SupervisorAgent

@asynccontextmanager
async def colab_lifespan(app: FastAPI):
    """Colab-optimized lifespan: uses Gemini API for all LLM workers."""
    print('[Colab] Initializing Gemini LLM Client...')
    llm_client = LLMClient(api_key=os.environ.get('GEMINI_API_KEY'))
    app.state.llm_client = llm_client

    print('[Colab] Initializing SupervisorAgent with LLM Synthesis enabled...')
    supervisor = SupervisorAgent(
        llm_client=llm_client,
        use_llm_synthesis=True,   # Full synthesis chain enabled
    )
    app.state.supervisor = supervisor

    # Print the runtime profile so you can verify workers are in LLM mode
    profile = supervisor.get_runtime_profile()
    print(f'[Colab] Worker modes: {profile["worker_modes"]}')
    print(f'[Colab] Synthesis mode: {profile["llm_synthesis_mode"]}')
    print('[Colab] ✅ System Ready — API is live!')

    yield

    await llm_client.aclose()
    print('[Colab] Shutdown complete.')

# Replace the default lifespan with the Colab-optimized one
app.router.lifespan_context = colab_lifespan

# Allow uvicorn to run inside a notebook event loop
nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8001)